# Amazon All Beauty — Time Series Analysis

**Data:** `amazon_beauty_full.csv.gz` (full 2000–2023 dataset)  
**Scope:** Global trend analysis uses 2000–2023 to justify 2015 cutoff; all modelling uses 2015+  
**Modules covered:** Time Series (decomposition, stationarity, ARIMA) + Cluster-level trends

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Load full dataset ─────────────────────────────────────────────────────────
df = pd.read_csv('amazon_beauty_full.csv.gz', compression='gzip',
                 parse_dates=['review_date'])

print(f'✓ Loaded: {len(df):,} rows, {df.shape[1]} columns')
print(f'  Date range: {df["review_year"].min()} – {df["review_year"].max()}')
print(f'  Avg rating: {df["rating"].mean():.2f}')
df.head(3)

## Step 1 — Annual Trends (full 2000–2023)

Kept full range intentionally: the pre-2015 sparsity visible here is the evidence for the 2015 cutoff applied in all downstream modelling.

In [ ]:
# ── Annual review volume and average rating (full range) ─────────────────────
annual = df.groupby('review_year').agg(
    review_count = ('rating', 'count'),
    avg_rating   = ('rating', 'mean')
).reset_index()

print(annual)

In [ ]:
# ── Plot annual trends — kept full 2000–2023 to show pre-2015 sparsity ───────
import os
os.makedirs('ts_figures', exist_ok=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle('Amazon All Beauty Reviews: Annual Trends (2000–2023)',
             fontsize=13, fontweight='bold')

axes[0].plot(annual['review_year'], annual['review_count'],
             color='cadetblue', linewidth=2, marker='o', markersize=4)
axes[0].set_title('Annual Review Volume')
axes[0].set_ylabel('Number of Reviews')
axes[0].grid(True, alpha=0.3)

axes[1].plot(annual['review_year'], annual['avg_rating'],
             color='darkslategray', linewidth=2, marker='o', markersize=4)
axes[1].set_title('Annual Average Rating')
axes[1].set_ylabel('Average Star Rating')
axes[1].set_ylim(3.5, 5.0)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ts_figures/01_annual_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 2 — Monthly Time Series (2015+)

All modelling starts from 2015. Pre-2015 data had fewer than 20,000 reviews per year — too sparse for reliable decomposition or forecasting.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# ── Build monthly time series — 2015+ only ────────────────────────────────────
# Pre-2015 excluded: annual counts <20K are too sparse for reliable modelling.
# Step 1 (above) retains full history to show the cutoff rationale visually.
monthly = (df[df['review_year'] >= 2015]
           .groupby(['review_year', 'review_month'])
           .agg(review_count = ('rating', 'count'),
                avg_rating   = ('rating', 'mean'))
           .reset_index())

monthly['date'] = pd.to_datetime(
    monthly['review_year'].astype(str) + '-' +
    monthly['review_month'].astype(str).str.zfill(2) + '-01')
monthly = monthly.set_index('date').asfreq('MS')

# Exclude incomplete September 2023 (only 9 days of data)
monthly = monthly[monthly.index <= '2023-08-01']

print(f'Monthly series: {len(monthly)} months')
print(f'Date range: {monthly.index.min().date()} → {monthly.index.max().date()}')
print(monthly.head())

In [ ]:
# ── Seasonal decomposition of review volume (2015+) ──────────────────────────
decomp_volume = seasonal_decompose(monthly['review_count'],
                                   model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))
fig.suptitle('Seasonal Decomposition — Monthly Review Volume (2015–2023)',
             fontsize=13, fontweight='bold')

axes[0].plot(monthly['review_count'], color='cadetblue')
axes[0].set_title('Observed'); axes[0].grid(True, alpha=0.3)

axes[1].plot(decomp_volume.trend, color='cadetblue')
axes[1].set_title('Trend'); axes[1].grid(True, alpha=0.3)

axes[2].plot(decomp_volume.seasonal, color='cadetblue')
axes[2].set_title('Seasonality'); axes[2].grid(True, alpha=0.3)

axes[3].plot(decomp_volume.resid, color='cadetblue')
axes[3].set_title('Residuals'); axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ts_figures/02_decomp_volume.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Seasonal decomposition of average rating (2015+) ─────────────────────────
decomp_rating = seasonal_decompose(monthly['avg_rating'],
                                   model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))
fig.suptitle('Seasonal Decomposition — Monthly Average Rating (2015–2023)',
             fontsize=13, fontweight='bold')

axes[0].plot(monthly['avg_rating'], color='darkslategray')
axes[0].set_title('Observed'); axes[0].grid(True, alpha=0.3)

axes[1].plot(decomp_rating.trend, color='darkslategray')
axes[1].set_title('Trend'); axes[1].grid(True, alpha=0.3)

axes[2].plot(decomp_rating.seasonal, color='darkslategray')
axes[2].set_title('Seasonality'); axes[2].grid(True, alpha=0.3)

axes[3].plot(decomp_rating.resid, color='darkslategray')
axes[3].set_title('Residuals'); axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ts_figures/03_decomp_rating.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Stationarity Tests

ADF and KPSS tests. Both series must be stationary before fitting ARIMA.

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

# ── ADF and KPSS tests ────────────────────────────────────────────────────────
for name, series in [('Review Volume', monthly['review_count']),
                     ('Average Rating', monthly['avg_rating'])]:
    adf_stat, adf_p = adfuller(series)[:2]
    kpss_stat, kpss_p = kpss(series)[:2]
    print(f'--- {name} ---')
    print(f'ADF  statistic: {adf_stat:.4f}  |  p-value: {adf_p:.4f}  '
          f'→ {"Stationary ✓" if adf_p < 0.05 else "Non-stationary ✗"}')
    print(f'KPSS statistic: {kpss_stat:.4f}  |  p-value: {kpss_p:.4f}  '
          f'→ {"Stationary ✓" if kpss_p > 0.05 else "Non-stationary ✗"}')
    print()

In [ ]:
# ── First-order differencing ──────────────────────────────────────────────────
monthly['volume_diff'] = monthly['review_count'].diff()
monthly['rating_diff'] = monthly['avg_rating'].diff()

for name, series in [('Review Volume (d=1)', monthly['volume_diff']),
                     ('Average Rating (d=1)', monthly['rating_diff'])]:
    series_clean = series.dropna()
    adf_stat, adf_p = adfuller(series_clean)[:2]
    kpss_stat, kpss_p = kpss(series_clean)[:2]
    print(f'--- {name} ---')
    print(f'ADF  p-value: {adf_p:.4f}  → {"Stationary ✓" if adf_p < 0.05 else "Non-stationary ✗"}')
    print(f'KPSS p-value: {kpss_p:.4f}  → {"Stationary ✓" if kpss_p > 0.05 else "Non-stationary ✗"}')
    print()

In [ ]:
# ── Second-order differencing ─────────────────────────────────────────────────
monthly['volume_diff2'] = monthly['review_count'].diff().diff()
monthly['rating_diff2'] = monthly['avg_rating'].diff().diff()

for name, series in [('Review Volume (d=2)', monthly['volume_diff2']),
                     ('Average Rating (d=2)', monthly['rating_diff2'])]:
    series_clean = series.dropna()
    adf_stat, adf_p = adfuller(series_clean)[:2]
    kpss_stat, kpss_p = kpss(series_clean)[:2]
    print(f'--- {name} ---')
    print(f'ADF  p-value: {adf_p:.4f}  → {"Stationary ✓" if adf_p < 0.05 else "Non-stationary ✗"}')
    print(f'KPSS p-value: {kpss_p:.4f}  → {"Stationary ✓" if kpss_p > 0.05 else "Non-stationary ✗"}')
    print()

In [ ]:
# ── Plot differenced series ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
fig.suptitle('Differenced Series (d=1)', fontsize=13, fontweight='bold')

axes[0].plot(monthly['volume_diff'], color='cadetblue', linewidth=1)
axes[0].set_title('Review Volume — First Difference')
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].grid(True, alpha=0.3)

axes[1].plot(monthly['rating_diff'], color='darkslategray', linewidth=1)
axes[1].set_title('Average Rating — First Difference')
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ts_figures/04_differenced_series.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 4 — ACF / PACF

Used to identify candidate p and q parameters for ARIMA.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# ── ACF and PACF — review volume (d=2) ───────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
fig.suptitle('ACF and PACF — Review Volume (d=2)', fontsize=13, fontweight='bold')
plot_acf(monthly['volume_diff2'].dropna(),  lags=24, ax=axes[0],
         color='cadetblue', title='ACF — Review Volume')
plot_pacf(monthly['volume_diff2'].dropna(), lags=24, ax=axes[1],
         color='cadetblue', title='PACF — Review Volume')
plt.tight_layout()
plt.savefig('ts_figures/05_acf_pacf_volume.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── ACF and PACF — average rating (d=2) ──────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
fig.suptitle('ACF and PACF — Average Rating (d=2)', fontsize=13, fontweight='bold')
plot_acf(monthly['rating_diff2'].dropna(),  lags=24, ax=axes[0],
         color='darkslategray', title='ACF — Average Rating')
plot_pacf(monthly['rating_diff2'].dropna(), lags=24, ax=axes[1],
         color='darkslategray', title='PACF — Average Rating')
plt.tight_layout()
plt.savefig('ts_figures/06_acf_pacf_rating.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 5 — ARIMA Grid Search

Select best p and q by AIC on the training set (hold out last 12 months as test).

In [ ]:
import itertools
from statsmodels.tsa.arima.model import ARIMA

# ── Train / test split — last 12 months as untouched test set ─────────────────
train_vol = monthly['review_count'].iloc[:-12]
test_vol  = monthly['review_count'].iloc[-12:]
train_rat = monthly['avg_rating'].iloc[:-12]
test_rat  = monthly['avg_rating'].iloc[-12:]

print(f'Train: {train_vol.index.min().date()} → {train_vol.index.max().date()} ({len(train_vol)} months)')
print(f'Test:  {test_vol.index.min().date()} → {test_vol.index.max().date()} ({len(test_vol)} months)')

In [ ]:
# ── Grid search — review volume ───────────────────────────────────────────────
results_vol = []
for p, q in itertools.product(range(0, 4), range(0, 4)):
    try:
        model = ARIMA(train_vol, order=(p, 2, q)).fit()
        results_vol.append({'p':p, 'd':2, 'q':q,
                            'AIC':round(model.aic, 2),
                            'BIC':round(model.bic, 2)})
    except:
        continue

df_vol_grid = pd.DataFrame(results_vol).sort_values('AIC')
print('=== Review Volume — Top 5 by AIC ===')
print(df_vol_grid.head())

In [ ]:
# ── Grid search — average rating ─────────────────────────────────────────────
results_rat = []
for p, q in itertools.product(range(0, 4), range(0, 4)):
    try:
        model = ARIMA(train_rat, order=(p, 2, q)).fit()
        results_rat.append({'p':p, 'd':2, 'q':q,
                            'AIC':round(model.aic, 2),
                            'BIC':round(model.bic, 2)})
    except:
        continue

df_rat_grid = pd.DataFrame(results_rat).sort_values('AIC')
print('=== Average Rating — Top 5 by AIC ===')
print(df_rat_grid.head())

## Step 6 — Fit Best Models & Residual Diagnostics

In [ ]:
# ── Fit best models ───────────────────────────────────────────────────────────
model_vol = ARIMA(train_vol, order=(2, 2, 3)).fit()
model_rat = ARIMA(train_rat, order=(1, 2, 1)).fit()

print('Review Volume — ARIMA(2,2,3)')
print(model_vol.summary())

In [ ]:
print('Average Rating — ARIMA(1,2,1)')
print(model_rat.summary())

In [ ]:
# ── Residual diagnostics — review volume ─────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
fig.suptitle('Residuals — ARIMA(2,2,3) Review Volume', fontsize=13, fontweight='bold')

residuals_vol = model_vol.resid
axes[0].plot(residuals_vol, color='cadetblue', linewidth=1)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Residuals over Time'); axes[0].grid(True, alpha=0.3)

axes[1].hist(residuals_vol, bins=30, color='cadetblue', edgecolor='white')
axes[1].set_title('Residual Distribution'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ts_figures/07_residuals_volume.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Residual diagnostics — average rating ────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 6))
fig.suptitle('Residuals — ARIMA(1,2,1) Average Rating', fontsize=13, fontweight='bold')

residuals_rat = model_rat.resid
axes[0].plot(residuals_rat, color='darkslategray', linewidth=1)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Residuals over Time'); axes[0].grid(True, alpha=0.3)

axes[1].hist(residuals_rat, bins=30, color='darkslategray', edgecolor='white')
axes[1].set_title('Residual Distribution'); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ts_figures/08_residuals_rating.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 7 — Forecast & Baseline Comparison

Compare ARIMA against Naive and Seasonal Naive baselines using RMSE.

In [ ]:
# ── Helper: RMSE without sklearn version dependency ──────────────────────────
def rmse(actual, predicted):
    return np.sqrt(np.mean((np.array(actual) - np.array(predicted))**2))

# ── Forecasts ─────────────────────────────────────────────────────────────────
forecast_vol = model_vol.forecast(steps=12)
forecast_rat = model_rat.forecast(steps=12)

# ── Baselines ─────────────────────────────────────────────────────────────────
naive_vol        = pd.Series([train_vol.iloc[-1]] * 12, index=test_vol.index)
seasonal_naive_vol = pd.Series(train_vol.iloc[-12:].values, index=test_vol.index)
naive_rat        = pd.Series([train_rat.iloc[-1]] * 12, index=test_rat.index)
seasonal_naive_rat = pd.Series(train_rat.iloc[-12:].values, index=test_rat.index)

# ── RMSE comparison ───────────────────────────────────────────────────────────
rmse_vol     = rmse(test_vol, forecast_vol)
rmse_naive_vol = rmse(test_vol, naive_vol)
rmse_sn_vol  = rmse(test_vol, seasonal_naive_vol)
rmse_rat     = rmse(test_rat, forecast_rat)
rmse_naive_rat = rmse(test_rat, naive_rat)
rmse_sn_rat  = rmse(test_rat, seasonal_naive_rat)

print('=== RMSE Comparison ===')
print(f"{'Model':<25} {'Review Volume':>15} {'Avg Rating':>12}")
print('-' * 55)
print(f"{'Naive':<25} {rmse_naive_vol:>15,.0f} {rmse_naive_rat:>12.4f}")
print(f"{'Seasonal Naive':<25} {rmse_sn_vol:>15,.0f} {rmse_sn_rat:>12.4f}")
print(f"{'ARIMA':<25} {rmse_vol:>15,.0f} {rmse_rat:>12.4f}")
print()
print('Note: for review volume, Seasonal Naive outperforms ARIMA.')
print('This is reported honestly — the volume series is highly irregular')
print('in 2022–2023, making it difficult for any model to beat a simple baseline.')

In [ ]:
# ── Final comparison plot ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle('ARIMA Forecast vs Baseline — Amazon All Beauty (2015+)',
             fontsize=13, fontweight='bold')

# Review volume
axes[0].plot(train_vol.index, train_vol, color='cadetblue', label='Train')
axes[0].plot(test_vol.index,  test_vol,  color='black',     label='Actual')
axes[0].plot(test_vol.index, forecast_vol,      color='coral',     linestyle='--',
             label=f'ARIMA(2,2,3)  RMSE={rmse_vol:,.0f}')
axes[0].plot(test_vol.index, naive_vol,          color='steelblue', linestyle=':',
             label=f'Naive  RMSE={rmse_naive_vol:,.0f}')
axes[0].plot(test_vol.index, seasonal_naive_vol, color='gray',      linestyle=':',
             label=f'Seasonal Naive  RMSE={rmse_sn_vol:,.0f}')
axes[0].set_title('Review Volume'); axes[0].set_ylabel('Number of Reviews')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Average rating
axes[1].plot(train_rat.index, train_rat, color='darkslategray', label='Train')
axes[1].plot(test_rat.index,  test_rat,  color='black',         label='Actual')
axes[1].plot(test_rat.index, forecast_rat,      color='coral',     linestyle='--',
             label=f'ARIMA(1,2,1)  RMSE={rmse_rat:.4f}')
axes[1].plot(test_rat.index, naive_rat,          color='steelblue', linestyle=':',
             label=f'Naive  RMSE={rmse_naive_rat:.4f}')
axes[1].plot(test_rat.index, seasonal_naive_rat, color='gray',      linestyle=':',
             label=f'Seasonal Naive  RMSE={rmse_sn_rat:.4f}')
axes[1].set_title('Average Rating'); axes[1].set_ylabel('Average Star Rating')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ts_figures/11_forecast_final_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 8 — Cluster-Level Time Series (2015+)

Break trends down by the four product clusters from the Clustering module.

In [ ]:
# ── Load cluster labels ───────────────────────────────────────────────────────
clusters = pd.read_csv('product_clusters_k4_2015plus_v3.csv',
                       usecols=['asin', 'cluster_name'])

print(f'Products with cluster labels: {len(clusters):,}')
print(clusters['cluster_name'].value_counts())

In [ ]:
# ── Join cluster labels to full review dataset ────────────────────────────────
df_clustered = df.merge(clusters, on='asin', how='inner')

print(f'Reviews matched to cluster: {len(df_clustered):,} '
      f'({len(df_clustered)/len(df)*100:.1f}% of total)')
print(f'Reviews without cluster:    {len(df) - len(df_clustered):,}')
print()
print(df_clustered['cluster_name'].value_counts())

In [ ]:
# ── Filter to 2015+ and exclude incomplete Sep 2023 ──────────────────────────
df_clustered = df_clustered[
    (df_clustered['review_year'] >= 2015) &
    ~((df_clustered['review_year'] == 2023) & (df_clustered['review_month'] == 9))
].copy()

print(f'Reviews after 2015 filter: {len(df_clustered):,}')
print()
print(df_clustered['cluster_name'].value_counts())

In [ ]:
# ── Monthly cluster aggregation ───────────────────────────────────────────────
cluster_monthly = (df_clustered
    .groupby(['cluster_name', 'review_year', 'review_month'])
    .agg(review_count = ('rating', 'count'),
         avg_rating   = ('rating', 'mean'))
    .reset_index())

cluster_monthly['date'] = pd.to_datetime(
    cluster_monthly['review_year'].astype(str) + '-' +
    cluster_monthly['review_month'].astype(str).str.zfill(2) + '-01')
cluster_monthly = cluster_monthly.sort_values(['cluster_name', 'date'])

print(f'✓ Monthly cluster data: {len(cluster_monthly):,} rows')
cluster_monthly.head()

In [ ]:
# ── Monthly trends by cluster ─────────────────────────────────────────────────
cluster_colors = {
    'Quiet Reliable Winners':             'cadetblue',
    'Budget / Polarizing':                'coral',
    'High-Rating / Sentiment-Mismatch':   'darkslategray',
    'High-Engagement / Low-Verification': 'goldenrod',
}

fig, axes = plt.subplots(2, 1, figsize=(12, 10))
fig.suptitle('Monthly Trends by Product Cluster (2015–2023)',
             fontsize=13, fontweight='bold')

for cluster, group in cluster_monthly.groupby('cluster_name'):
    axes[0].plot(group['date'], group['avg_rating'],
                 label=cluster, color=cluster_colors[cluster], linewidth=1.5)
axes[0].set_title('Average Rating by Cluster')
axes[0].set_ylabel('Average Star Rating')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

for cluster, group in cluster_monthly.groupby('cluster_name'):
    axes[1].plot(group['date'], group['review_count'],
                 label=cluster, color=cluster_colors[cluster], linewidth=1.5)
axes[1].set_title('Monthly Review Volume by Cluster')
axes[1].set_ylabel('Number of Reviews')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ts_figures/12_cluster_monthly_trends.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Annual cluster trends ─────────────────────────────────────────────────────
cluster_annual = (df_clustered
    .groupby(['cluster_name', 'review_year'])
    .agg(review_count = ('rating', 'count'),
         avg_rating   = ('rating', 'mean'))
    .reset_index())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Annual Trends by Product Cluster (2015–2023)',
             fontsize=13, fontweight='bold')

for cluster, group in cluster_annual.groupby('cluster_name'):
    axes[0].plot(group['review_year'], group['avg_rating'],
                 label=cluster, color=cluster_colors[cluster],
                 linewidth=2, marker='o', markersize=4)
axes[0].set_title('Average Rating by Cluster')
axes[0].set_ylabel('Average Star Rating')
axes[0].set_xlabel('Year')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

for cluster, group in cluster_annual.groupby('cluster_name'):
    axes[1].plot(group['review_year'], group['review_count'],
                 label=cluster, color=cluster_colors[cluster],
                 linewidth=2, marker='o', markersize=4)
axes[1].set_title('Annual Review Volume by Cluster')
axes[1].set_ylabel('Number of Reviews')
axes[1].set_xlabel('Year')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('ts_figures/13_cluster_annual_trends.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 9 — Save All Figures

In [ ]:
import shutil
shutil.make_archive('ts_figures', 'zip', 'ts_figures')
print('✓ All figures saved to ts_figures.zip')